[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/en_clase/S11_03_deteccion_segmentacion_en_clase.ipynb)

# S11 · 03 — Detección de objetos y segmentación · versión en clase

⏱ 35 min en clase · Sprint 11 · UD 4.6

**Contexto TurisData.** Clasificar una foto dice *qué es*; pero al cliente le interesan preguntas como *"¿cuántas sombrillas hay en la playa del resort?"* (detección) o *"¿qué parte de la imagen es piscina?"* (segmentación). En esta práctica verás los conceptos, las métricas (IoU, NMS, mAP) con números pequeños y un ejemplo completo **sin internet**, con imágenes aéreas sintéticas de una playa y con OpenCV.

**Este cuaderno tiene 2 ejercicios** (1 a 2), los que se hacen en el aula. Las métricas (precisión, recall, AP y mAP), el detector clásico con OpenCV, la segmentación, la rama YOLO, el mini-reto del contador de sombrillas y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1 y 2 son los 2 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os, sys, subprocess
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw

try:
    import cv2
except ImportError:      # en Colab y en el taller ya viene instalado; esta celda solo actúa si faltara
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"])
    import cv2

SEMILLA = 42
print("OpenCV", cv2.__version__)
print("Listo ✔")

### ✏️ A2 · EJERCICIO 1 de 2 — La IoU de dos cajas

**Tu turno:** Escribe `iou(a, b)` que reciba dos cajas `[x1, y1, x2, y2]` y devuelva su IoU (un número entre 0 y 1). Cuidado con el caso en que no se solapan (la intersección vale 0, no negativa).

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A2).

1. ¿Cuánto vale la IoU de `[0,0,10,10]` y `[5,5,15,15]`?
2. ¿Por qué se acota con `max(0, ...)`?
3. ¿Cuándo se considera correcta una detección?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A2, **después de intentarlo**.

In [ ]:
# pista: ancho y alto de la intersección con max(0, ...); unión = área_a + área_b - intersección
def iou(a, b):
    ...

assert abs(iou([0, 0, 10, 10], [5, 5, 15, 15]) - 25 / 175) < 1e-9, "IoU esperada 25/175 ≈ 0.143"
assert iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0, "Cajas idénticas: IoU = 1"
assert iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0, "Cajas separadas: IoU = 0 (¿has acotado con max(0, ...)?)"
assert abs(iou([0, 0, 10, 10], [0, 0, 10, 5]) - 0.5) < 1e-9, "Una caja dentro de otra: IoU = área pequeña / área grande"
print("✔ Ejercicio 1 correcto")

### ✏️ A3 · EJERCICIO 2 de 2 — NMS: quedarse con una caja por objeto

**Tu turno:** Escribe `nms(cajas, puntuaciones, umbral_iou=0.5)` que devuelva la **lista de índices** de las cajas que se conservan, en orden de confianza decreciente. Usa tu función `iou`.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué cajas conserva NMS en el ejemplo con umbral 0,5?
2. ¿Qué pasa con un umbral de 0,99?
3. ¿Por qué un detector propone cajas duplicadas?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: ordena índices por puntuación; mientras queden, guarda el primero y filtra los que tienen iou > umbral con él
def nms(cajas, puntuaciones, umbral_iou=0.5):
    ...

_cajas = [[10, 10, 50, 50], [12, 12, 52, 52], [100, 100, 140, 140], [11, 9, 49, 51]]
_punt = [0.9, 0.8, 0.7, 0.6]
assert nms(_cajas, _punt) == [0, 2], "Debe quedar la caja 0 (mejor) y la 2 (lejana); la 1 y la 3 son duplicados de la 0"
assert nms(_cajas, _punt, umbral_iou=0.99) == [0, 1, 2, 3], "Con un umbral muy alto no se descarta nada"
assert nms([[0, 0, 5, 5]], [0.3]) == [0], "Con una sola caja se conserva"
print("✔ Ejercicio 2 correcto")

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_03_deteccion_segmentacion.ipynb)** trae las métricas AP y mAP, el detector clásico con OpenCV, la segmentación, la rama YOLO, el mini-reto del contador de sombrillas y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `numpy`, `matplotlib`, `PIL` (para dibujar imágenes) y `cv2` (OpenCV, la librería de visión clásica). Si `cv2` faltara, la celda lo instala.
    - `os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"` silencia los mensajes técnicos de TensorFlow, que se usará más adelante.
- **Qué debe salir:**
    - `OpenCV x.y.z` (la versión instalada, por ejemplo 5.0.0) y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 2. Cajas y la IoU

#### A2 · Ejercicio 1 — La IoU de dos cajas

**Contexto TurisData.** Clasificar una foto dice *qué es*; pero al cliente le interesan preguntas como *«¿cuántas sombrillas hay en la playa del resort?»* (detección) o *«¿qué parte de la imagen es piscina?»* (segmentación). En esta práctica verás los conceptos, las métricas (IoU, NMS, mAP) con números pequeños y un ejemplo completo **sin internet**, con imágenes aéreas sintéticas de una playa y con OpenCV.

| Tarea | Pregunta | Salida | Ejemplo TurisData |
|---|---|---|---|
| **Clasificación** | ¿Qué hay en la foto? | una etiqueta | "esta foto es de una piscina" |
| **Detección** | ¿Qué hay y **dónde**? | cajas (x1, y1, x2, y2) + clase + confianza | contar sombrillas en la playa |
| **Segmentación semántica** | ¿A qué clase pertenece **cada píxel**? | una máscara | superficie de piscina en la foto |
| **Segmentación de instancias** | Lo mismo, separando cada objeto | una máscara por objeto | cada tumbona por separado |

Los detectores modernos (YOLO, Faster R-CNN, DETR) son redes convolucionales que predicen muchas cajas candidatas de una vez;
después se limpian con **NMS** y se evalúan con **mAP**. Empezamos por esas piezas.

**Idea.** Una caja se escribe `[x1, y1, x2, y2]` (esquina superior izquierda y esquina inferior derecha, en píxeles).
Para saber si una caja predicha "acierta" con la real, se mide cuánto se solapan:

`IoU = área de la intersección / área de la unión`

0 = no se tocan; 1 = idénticas. Por convención, una detección es correcta (verdadero positivo) si su IoU con una caja real es **≥ 0,5**.

**Ejemplo:** cajas `[0,0,10,10]` y `[5,5,15,15]`: intersección 5×5 = 25; unión 100 + 100 − 25 = 175; IoU = 25/175 ≈ 0,143.

- **Qué hace:**
    - `ancho` y `alto` son las dimensiones de la **intersección** de las dos cajas: `min` de los extremos derechos menos `max` de los izquierdos (y lo mismo en vertical), acotados con `max(0, ...)` para que no salgan negativos si las cajas no se tocan.
    - `inter` es el área de la intersección; `area_a` y `area_b` las de cada caja; la unión es `area_a + area_b - inter`.
    - `iou` devuelve `inter / union` (o 0 si la unión es 0). Las comprobaciones prueban solape parcial (25/175 ≈ 0,143), cajas idénticas (1), cajas separadas (0) y una caja dentro de otra (0,5).
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Cómo leerlo:**
    - La IoU va de 0 (no se tocan) a 1 (idénticas). Por convención, una detección es correcta (verdadero positivo) si su IoU con una caja real es **≥ 0,5**.
- **Si falla:**
    - `IoU esperada 25/175 ≈ 0.143` → la unión o la intersección están mal calculadas (recuerda restar la intersección de la suma de áreas).
    - `Cajas separadas: IoU = 0 (¿has acotado con max(0, ...)?)` → falta acotar `ancho` y `alto` con `max(0, ...)`.
    - `NameError: name 'iou' is not defined` → no has ejecutado la celda del ejercicio.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale la IoU de `[0,0,10,10]` y `[5,5,15,15]`?* → **25 / 175 ≈ 0,143**: intersección 5×5 = 25 y unión 100 + 100 − 25 = 175.
2. *¿Por qué se acota con `max(0, ...)`?* → Para que, si las cajas **no se solapan**, la intersección valga 0 y no un número negativo.
3. *¿Cuándo se considera correcta una detección?* → Cuando su IoU con una caja real es **≥ 0,5**.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def iou(a, b):
    ancho = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    alto = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ancho * alto
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. NMS: una caja por objeto

#### A3 · Ejercicio 2 — NMS: quedarse con una caja por objeto

**Idea.** Un detector suele proponer **varias cajas casi iguales** para el mismo objeto. NMS las limpia:
1. Ordena las cajas por confianza (de mayor a menor).
2. Se queda con la primera.
3. **Descarta** las demás que se solapan con ella más de un umbral (por ejemplo IoU > 0,5).
4. Repite con las que quedan.

**Ejemplo:** cajas A (confianza 0,9), B (0,8, casi igual a A) y C (0,7, lejos). NMS conserva A y C: B era un duplicado de A.

- **Qué hace:**
    - `np.argsort(puntuaciones)[::-1]` ordena los índices de las cajas por confianza, de mayor a menor (`orden`).
    - Mientras queden índices, `orden.pop(0)` toma el de mayor confianza (`mejor`), lo guarda en `conservadas` y se queda en `orden` solo con las cajas cuyo `iou` con `mejor` es **menor o igual** que `umbral_iou`.
    - Las comprobaciones usan 4 cajas: la 0 (confianza 0,9), la 1 y la 3 (casi iguales a la 0) y la 2 (lejana).
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Con el umbral de 0,5 se conservan las cajas `[0, 2]`: la 1 y la 3 eran duplicados de la 0. Con un umbral de 0,99 no se descarta ninguna: `[0, 1, 2, 3]`.
- **Si falla:**
    - `Debe quedar la caja 0 (mejor) y la 2 (lejana)...` → no filtras las cajas que se solapan con la elegida, o filtras con la condición al revés.
    - `Con un umbral muy alto no se descarta nada` → el umbral no se usa o se compara mal (`<=` frente a `<`).
    - `NameError: name 'iou' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué cajas conserva NMS en el ejemplo con umbral 0,5?* → **[0, 2]**: la mejor y la lejana.
2. *¿Qué pasa con un umbral de 0,99?* → Casi nada se considera duplicado: se conservan las **4** cajas.
3. *¿Por qué un detector propone cajas duplicadas?* → Porque predice **muchas cajas candidatas** a la vez, y varias caen casi en el mismo objeto.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def nms(cajas, puntuaciones, umbral_iou=0.5):
    orden = list(np.argsort(puntuaciones)[::-1])
    conservadas = []
    while orden:
        mejor = orden.pop(0)
        conservadas.append(int(mejor))
        orden = [i for i in orden if iou(cajas[mejor], cajas[i]) <= umbral_iou]
    return conservadas
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>